# 183. Customers Who Never Order

## Problem Description

A customer database contains registered customers and their purchase records.

### Table Schema

**Customers**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| id | int | customer identifier (key) |
| name | varchar | customer name |

**Orders**

| Column | Type | Meaning / constraint |
| --- | --- | --- |
| id | int | order identifier (key) |
| customerId | int | reference to Customers.id |

### Task

Find every registered customer whose ID does not appear in Orders.customerId. Return their names in a single column named Customers. Different customers may have identical names; do not eliminate them merely because the names match. Any row order is accepted.

## Example

### Input

**customers**

| id | name |
| --- | --- |
| 1 | A |
| 2 | B |
| 3 | A |
| 4 | D |

**orders**

| id | customerId |
| --- | --- |
| 1 | 1 |
| 2 | 1 |
| 3 | 4 |

### Expected Output

| Customers |
| --- |
| B |
| A |

### Explanation

Customers 1 and 4 have orders; customers 2 and 3 have none. The latter two names are B and A.

## Practice

The next code cell constructs exactly these example DataFrames. Try the blank practice cell before reading the solutions.

## References

The problem is restated for self-contained practice; the examples are original. Community links are for further reading, and implementations below are independently written.

[Original problem and schema (Doocs)](https://github.com/doocs/leetcode/blob/main/solution/0100-0199/0183.Customers%20Who%20Never%20Order/README_EN.md)

[Community solutions index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) — further reading; implementations below are independently written.

[LeetCode problem](https://leetcode.com/problems/customers-who-never-order/) · [Public solution discussions](https://leetcode.com/problems/customers-who-never-order/solutions/)

In [ ]:
import pandas as pd
import re
from pandas.testing import assert_frame_equal
customers = pd.DataFrame([[1,"A"],[2,"B"],[3,"A"],[4,"D"]], columns=["id","name"])
orders = pd.DataFrame([[1,1],[2,1],[3,4]], columns=["id","customerId"])
expected = pd.DataFrame([["B"],["A"]], columns=["Customers"])
display(customers)
display(orders)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions
Compare the approaches using the same input.

### Solution 1 — Anti-filter
以 ID 判斷，不以名字判斷。

In [ ]:
def solution_1(customers, orders):
    return customers.loc[~customers.id.isin(orders.customerId),['name']].rename(columns={'name':'Customers'})

result_1 = solution_1(customers.copy(), orders.copy())
display(result_1)

### Solution 2 — Left anti-join
indicator 可診斷資料是否匹配。

In [ ]:
def solution_2(customers, orders):
    m = customers.merge(orders[['customerId']].drop_duplicates(),left_on='id',right_on='customerId',how='left',indicator=True)
    return m.loc[m['_merge'].eq('left_only'),['name']].rename(columns={'name':'Customers'})

result_2 = solution_2(customers.copy(), orders.copy())
display(result_2)

### Solution 3 — Set difference
集合操作；保留原始客戶粒度。

In [ ]:
def solution_3(customers, orders):
    ids = set(customers.id)-set(orders.customerId)
    return customers.loc[customers.id.isin(ids),['name']].rename(columns={'name':'Customers'})

result_3 = solution_3(customers.copy(), orders.copy())
display(result_3)

## Check all solutions
Normalize row order for these examples.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Which method is easiest to read, debug, and extend? Try a zero-match input and duplicate keys where permitted.